<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>VAE on MNIST</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/8a860b4/examples/vae_mnist.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/vae_mnist.ipynb)

</div>

---

Training a variational autoencoder on [MNIST](http://yann.lecun.com/exdb/mnist/) using [Ion](https://github.com/auxeno/ion).

In [1]:
# !pip install ion-nn optax plotly tqdm

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## Load MNIST

In [3]:
import urllib.request
from pathlib import Path

from jaxtyping import UInt8

def load_mnist() -> tuple[
    UInt8[np.ndarray, "60000 28 28 1"],
    UInt8[np.ndarray, " 60000"],
    UInt8[np.ndarray, "10000 28 28 1"],
    UInt8[np.ndarray, " 10000"],
]:
    """Download MNIST dataset and return (train_images, train_labels, test_images, test_labels)."""
    cache_dir = Path.home() / ".cache" / "ion" / "mnist"
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / "mnist.npz"

    if not path.exists():
        print("Downloading MNIST...")
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
            path,
        )

    data = np.load(path)

    # Add channels singleton dim to images
    train_images = data["x_train"][..., None]
    train_labels = data["y_train"]
    test_images = data["x_test"][..., None]
    test_labels = data["y_test"]

    return train_images, train_labels, test_images, test_labels

images, _, _, _ = load_mnist()
print(f"Shape: {images.shape}")
print(f"Dtype: {images.dtype}")

Shape: (60000, 28, 28, 1)
Dtype: uint8


In [4]:
from plotly.subplots import make_subplots

num_rows, num_cols = 2, 6
img_size = 128

rng = np.random.default_rng(0)
sample_indices = rng.choice(len(images), size=num_rows * num_cols, replace=False)
samples = images[sample_indices, :, :, 0]

fig = make_subplots(rows=num_rows, cols=num_cols, horizontal_spacing=0.02, vertical_spacing=0.02)
for i, img in enumerate(samples):
    row, col = divmod(i, num_cols)
    fig.add_trace(go.Image(z=np.stack([img] * 3, axis=-1)), row=row + 1, col=col + 1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(
    height=num_rows * img_size,
    width=num_cols * img_size,
    title="MNIST samples",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

## Building the VAE

A VAE learns a compressed latent representation by training an encoder and decoder jointly. The encoder maps an image to a distribution over latent vectors (parameterized by a mean and log-variance), the decoder reconstructs images from samples of that distribution. The loss combines a reconstruction term (binary cross-entropy on pixels normalized to [0, 1]) with a KL divergence term that regularizes the latent space toward a standard normal.

We use `latent_dim=2` so we can directly visualize the latent space as a 2D grid of decoded images.

In [5]:
from jaxtyping import Array, Float, PRNGKeyArray


class Encoder(nn.Module):
    conv_1: nn.Conv
    conv_2: nn.Conv
    dense: nn.Linear
    dense_mean: nn.Linear
    dense_logvar: nn.Linear

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key, 5)
        self.conv_1 = nn.Conv(1, 16, (4, 4), stride=2, padding=1, key=keys[0])
        self.conv_2 = nn.Conv(16, 32, (4, 4), stride=2, padding=1, key=keys[1])
        self.dense = nn.Linear(7 * 7 * 32, 64, key=keys[2])
        self.dense_mean = nn.Linear(64, 2, key=keys[3])
        self.dense_logvar = nn.Linear(64, 2, key=keys[4])

    def __call__(
        self, 
        x: Float[Array, "b 28 28 1"]
    ) -> tuple[Float[Array, "b 2"], Float[Array, "b 2"]]:
        
        x = jax.nn.relu(self.conv_1(x))
        x = jax.nn.relu(self.conv_2(x))

        x = x.reshape(-1, 7 * 7 * 32)

        x = jax.nn.relu(self.dense(x))
        mean = self.dense_mean(x)
        logvar = self.dense_logvar(x)

        return mean, logvar
    

class Decoder(nn.Module):
    dense: nn.Linear
    conv_t_1: nn.ConvTranspose
    conv_t_2: nn.ConvTranspose

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key, 3)
        self.dense = nn.Linear(2, 7 * 7 * 32, key=keys[0])
        self.conv_t_1 = nn.ConvTranspose(32, 16, (4, 4), stride=2, padding="SAME", key=keys[1])
        self.conv_t_2 = nn.ConvTranspose(16, 1, (4, 4), stride=2, padding="SAME", key=keys[2])
    
    def __call__(self, x: Float[Array, "b 2"]) -> Float[Array, "b 28 28 1"]:

        x = jax.nn.relu(self.dense(x))

        x = x.reshape(-1, 7, 7, 32)

        x = jax.nn.relu(self.conv_t_1(x))
        x = self.conv_t_2(x)

        return x


class VAE(nn.Module):
    encoder: Encoder
    decoder: Decoder

    def __init__(self, *, key: PRNGKeyArray) -> None:
        keys = jax.random.split(key)
        self.encoder = Encoder(key=keys[0])
        self.decoder = Decoder(key=keys[1])

    def __call__(
        self, 
        x: Float[Array, "b 28 28 1"], 
        *, 
        key: PRNGKeyArray
    ) -> Float[Array, "b 28 28 1"]:

        mean, logvar = self.encoder(x)
        
        z = self.reparametrize(mean, logvar, key=key)
        
        x = self.decoder(z)
        
        return x

    @staticmethod
    def reparametrize(
        mean: Float[Array, "b 2"], 
        logvar: Float[Array, "b 2"],
        *,  
        key: PRNGKeyArray,
    ) -> Float[Array, "b 2"]:
        
        std = jnp.exp(0.5 * logvar)
        eps = jax.random.normal(shape=mean.shape, key=key)
        z = mean + std * eps

        return z


model = VAE(key=jax.random.key(0))
print("Number of parameters: ", model.num_params)
model

Number of parameters:  122341


VAE(
  encoder=Encoder(
    conv_1=Conv(
      w=Param(f32[4, 4, 1, 16], trainable=True),
      b=Param(f32[16], trainable=True),
      kernel_shape=(4, 4),
      stride=(2, 2),
      padding=((1, 1), (1, 1)),
      dilation=(1, 1),
      groups=1,
    ),
    conv_2=Conv(
      w=Param(f32[4, 4, 16, 32], trainable=True),
      b=Param(f32[32], trainable=True),
      kernel_shape=(4, 4),
      stride=(2, 2),
      padding=((1, 1), (1, 1)),
      dilation=(1, 1),
      groups=1,
    ),
    dense=Linear(
      w=Param(f32[1568, 64], trainable=True),
      b=Param(f32[64], trainable=True),
    ),
    dense_mean=Linear(
      w=Param(f32[64, 2], trainable=True),
      b=Param(f32[2], trainable=True),
    ),
    dense_logvar=Linear(
      w=Param(f32[64, 2], trainable=True),
      b=Param(f32[2], trainable=True),
    ),
  ),
  decoder=Decoder(
    dense=Linear(
      w=Param(f32[2, 1568], trainable=True),
      b=Param(f32[1568], trainable=True),
    ),
    conv_t_1=ConvTranspose(
      w=Param(f32[4, 4, 32, 16], trainable=True),
      b=Param(f32[16], trainable=True),
      kernel_shape=(4, 4),
      stride=(2, 2),
      padding=((2, 2), (2, 2)),
      dilation=(1, 1),
      groups=1,
    ),
    conv_t_2=ConvTranspose(
      w=Param(f32[4, 4, 16, 1], trainable=True),
      b=Param(f32[1], trainable=True),
      kernel_shape=(4, 4),
      stride=(2, 2),
      padding=((2, 2), (2, 2)),
      dilation=(1, 1),
      groups=1,
    ),
  ),
)

In [6]:
dummy_input = jnp.ones((5, 28, 28, 1))
logits = model(dummy_input, key=jax.random.key(0))
jax.nn.sigmoid(logits).squeeze()

Array([[[0.56696063, 0.42183325, 0.56339294, ..., 0.579975  ,
         0.52452224, 0.42534664],
        [0.53176004, 0.5265584 , 0.3271172 , ..., 0.54083204,
         0.4203208 , 0.47498003],
        [0.58008957, 0.56238335, 0.646634  , ..., 0.46069753,
         0.50977796, 0.39145964],
        ...,
        [0.53705466, 0.46694618, 0.45443916, ..., 0.46177122,
         0.34634462, 0.4714611 ],
        [0.65859056, 0.41631636, 0.49499458, ..., 0.49018633,
         0.6279973 , 0.46093896],
        [0.5297603 , 0.53957796, 0.42277673, ..., 0.48991668,
         0.5394716 , 0.49860877]],

       [[0.50445807, 0.4873851 , 0.5146868 , ..., 0.5041953 ,
         0.50665724, 0.49004352],
        [0.5063906 , 0.49943203, 0.49225256, ..., 0.503187  ,
         0.48964158, 0.50895756],
        [0.4990475 , 0.50978005, 0.5168828 , ..., 0.49698865,
         0.509051  , 0.49594465],
        ...,
        [0.505712  , 0.5008001 , 0.49768227, ..., 0.49536425,
         0.47528133, 0.49468717],
        [0.5141984 , 0.4917218 , 0.501743  , ..., 0.4967511 ,
         0.5193994 , 0.49951506],
        [0.5044363 , 0.5030272 , 0.49723765, ..., 0.5002902 ,
         0.5046699 , 0.50519896]],

       [[0.51238054, 0.4796765 , 0.55675256, ..., 0.5243608 ,
         0.5193104 , 0.48028478],
        [0.54413784, 0.43851033, 0.4436166 , ..., 0.43170497,
         0.44633436, 0.5184496 ],
        [0.4906599 , 0.51850754, 0.51726747, ..., 0.50192356,
         0.54404485, 0.5012828 ],
        ...,
        [0.48480535, 0.5095848 , 0.43076387, ..., 0.54477423,
         0.3944091 , 0.47874433],
        [0.52119786, 0.43368232, 0.5392046 , ..., 0.5177153 ,
         0.5950278 , 0.4770171 ],
        [0.5221404 , 0.49392042, 0.5008398 , ..., 0.4912674 ,
         0.47697484, 0.5060492 ]],

       [[0.51675427, 0.49551117, 0.49988243, ..., 0.51356065,
         0.50170255, 0.49003774],
        [0.5042177 , 0.5020651 , 0.44003642, ..., 0.50468093,
         0.5085034 , 0.48964489],
        [0.5254993 , 0.49691406, 0.52130115, ..., 0.4857064 ,
         0.4971588 , 0.46119547],
        ...,
        [0.5104167 , 0.47642124, 0.4839788 , ..., 0.4834504 ,
         0.48632804, 0.49793032],
        [0.5414367 , 0.48197487, 0.49307713, ..., 0.50432354,
         0.52522564, 0.47336525],
        [0.50430924, 0.5053616 , 0.47241515, ..., 0.48893553,
         0.49706212, 0.49123293]],

       [[0.5151874 , 0.47085205, 0.57624274, ..., 0.52492964,
         0.5263068 , 0.47085482],
        [0.55770457, 0.43179476, 0.43841755, ..., 0.43430796,
         0.4588557 , 0.53944737],
        [0.47962552, 0.5417207 , 0.51283103, ..., 0.4696633 ,
         0.5615832 , 0.52240115],
        ...,
        [0.4862137 , 0.5070671 , 0.43881974, ..., 0.54383606,
         0.3600501 , 0.47264662],
        [0.5354616 , 0.41763574, 0.53530383, ..., 0.5152123 ,
         0.6215107 , 0.47198117],
        [0.53171295, 0.49786246, 0.50326043, ..., 0.49297774,
         0.47601303, 0.517761  ]]], dtype=float32)

## Training

The VAE loss is the sum of two terms:

- **Reconstruction loss**: binary cross-entropy between the normalized input and the decoder output, measuring how well the model reconstructs the original image.
- **KL divergence**: regularizes the latent distribution toward a standard normal, preventing the encoder from collapsing to point estimates.

We track both terms separately to monitor training.

In [7]:
from tqdm import tqdm


def loss_fn(
    model: VAE,
    x: Float[Array, "b 28 28 1"],
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], tuple[Float[Array, ""], Float[Array, ""]]]:
    """Returns (total_loss, (loss_recon, loss_kl))."""

    mean, logvar = model.encoder(x)
    z = model.reparametrize(mean, logvar, key=key)
    logits = model.decoder(z)

    # Binary cross-entropy loss over pixels
    bce_pixel = optax.sigmoid_binary_cross_entropy(logits, x)

    # Sum errors per image then average across the batch
    loss_recon = bce_pixel.sum(axis=(1, 2, 3)).mean()

    # KL divergence of each latent vector dim from the standard normal distribution
    z_kl = -0.5 * (1 + logvar - mean ** 2 - jnp.exp(logvar))

    # Sum for total KL divergence of each vector then average across the batch
    loss_kl = z_kl.sum(axis=-1).mean()

    loss = loss_recon + loss_kl

    return loss, (loss_recon, loss_kl)


@jax.jit
def train_step(model, optimizer, batch, key):
    grads, (loss_recon, loss_kl) = jax.grad(loss_fn, has_aux=True)(model, batch, key)         
    model, optimizer = optimizer.update(model, grads)
    return model, optimizer, loss_recon, loss_kl


LEARNING_RATE = 5e-4
BATCH_SIZE = 64
NUM_EPOCHS = 10

model = VAE(key=jax.random.key(0))
optimizer = ion.Optimizer(optax.adam(LEARNING_RATE), model)

num_batches = len(images) // BATCH_SIZE

recon_losses = []
kl_losses = []

rng = jax.random.key(0)

for epoch in range(NUM_EPOCHS):
    
    rng, key_shuffle = jax.random.split(rng)
    indices = jax.random.permutation(key_shuffle, len(images))

    epoch_recon, epoch_kl = 0.0, 0.0
    for i in tqdm(range(num_batches), desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
        batch_indices = indices[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        batch = jnp.asarray(images[batch_indices], dtype=jnp.float32) / 255.0

        rng, key_step = jax.random.split(rng)
        model, optimizer, loss_recon, loss_kl = train_step(model, optimizer, batch, key_step)

        epoch_recon += loss_recon.item()
        epoch_kl += loss_kl.item()

    recon_losses.append(epoch_recon / num_batches)
    kl_losses.append(epoch_kl / num_batches)
    print(f"  recon loss: {recon_losses[-1]:.2f}  kl loss: {kl_losses[-1]:.2f}")

Epoch 1/10: 100%|██████████| 937/937 [00:13<00:00, 68.70it/s]


  recon loss: 202.51  kl loss: 7.50


Epoch 2/10: 100%|██████████| 937/937 [00:13<00:00, 69.02it/s]


  recon loss: 170.95  kl loss: 5.73


Epoch 3/10: 100%|██████████| 937/937 [00:15<00:00, 58.77it/s]


...

  recon loss: 152.41  kl loss: 6.14


Epoch 9/10: 100%|██████████| 937/937 [00:18<00:00, 50.62it/s]


  recon loss: 151.38  kl loss: 6.21


Epoch 10/10: 100%|██████████| 937/937 [00:22<00:00, 41.93it/s]

  recon loss: 150.49  kl loss: 6.24


In [8]:
total_losses = [r + k for r, k in zip(recon_losses, kl_losses)]

fig = go.Figure()
fig.add_trace(go.Scatter(y=total_losses, mode="lines", name="Total", line=dict(width=5, color="#636EFA"), opacity=0.8))
fig.add_trace(go.Scatter(y=recon_losses, mode="lines", name="Reconstruction", line=dict(width=5, color="#EF553B"), opacity=0.8))
fig.add_trace(go.Scatter(y=kl_losses, mode="lines", name="KL divergence", line=dict(width=5, color="#00CC96"), opacity=0.8))
fig.update_layout(title="Training loss", xaxis_title="Epoch", yaxis_title="Loss", height=400, template="plotly_white")
fig.show()

## Visualizing the latent space

With a 2D latent space we can visualize the decoder's output across the entire space. We sample a uniform grid of latent vectors and decode each one.

In [9]:
n = 20
grid_range = np.linspace(-2.0, 2.0, n)

@jax.jit
def decode_grid(z_batch):
    logits = model.decoder(z_batch)
    return jax.nn.sigmoid(logits)[:, :, :, 0]

# Build grid of latent vectors and decode in one batch
z_grid = np.array([[z1, z2] for z2 in reversed(grid_range) for z1 in grid_range], dtype=np.float32)
decoded = np.array(decode_grid(jnp.array(z_grid)))

# Stitch into a single image
mosaic = np.zeros((n * 28, n * 28))
for idx in range(n * n):
    row, col = divmod(idx, n)
    mosaic[row * 28 : (row + 1) * 28, col * 28 : (col + 1) * 28] = decoded[idx]

# Convert to grayscale RGB uint8 to avoid storing raw floats as JSON
mosaic_uint8 = (mosaic * 255).astype(np.uint8)
mosaic_rgb = np.stack([mosaic_uint8] * 3, axis=-1)

fig = go.Figure(data=go.Image(z=mosaic_rgb))
fig.update_layout(
    height=600,
    width=600,
    template="plotly_white",
    title="Latent space",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False, scaleanchor="x"),
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()